In [82]:
import pandas as pd
import mysql.connector

# Connect to MySQL database
def get_db_connection():
    mydb = mysql.connector.connect(
        host="alvcantu.mysql.pythonanywhere-services.com",
        user="alvcantu",
        password="h63Efp09-d",
        database="alvcantu$default"
    )
    cursor = mydb.cursor(dictionary=True) # Outputs as dictionary
    return mydb, cursor

def execute_query(query):
    mydb, cursor = get_db_connection()
    cursor.execute(query)
    # Fetch all rows as a list of dictionaries where keys are column names
    rows = cursor.fetchall()
    
    # Convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(rows)
    
    # Close the connection
    cursor.close()
    mydb.close()
    
    return df

query = '''
SELECT 
    t.TransactionID AS TransactionID,
    t.InvoiceID AS InvoiceID,
    i.CustomerID AS CustomerID,
    i.InvoiceDate AS InvoiceDate,
    i.Country AS Country,
    t.StockCode AS StockCode,
    t.Description AS Description,
    t.Quantity AS Quantity,
    t.UnitPrice AS UnitPrice,
    (t.Quantity * t.UnitPrice) AS Sales
FROM 
    ONR_FactTransactions t
JOIN 
    ONR_DimInvoice i ON t.InvoiceID = i.InvoiceID;
'''

df = execute_query(query)

In [83]:
from sklearn.preprocessing import LabelEncoder
from datetime import datetime

# Convert to valid data types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['UnitPrice'] = df['UnitPrice'].fillna(0).astype('float32')
# Feature Engineering
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['UnitPrice'] = df['UnitPrice'].fillna(0).astype('float32')

# Encode categorical variables
le = LabelEncoder()
df['CustomerID'] = le.fit_transform(df['CustomerID'].astype(str))
df['StockCode'] = le.fit_transform(df['StockCode'].astype(str))
df['Country'] = le.fit_transform(df['Country'])

# Features the model was trained on
features = ['CustomerID', 'StockCode', 'Year', 'Month', 'Day', 'DayOfWeek', 'UnitPrice', 'Quantity', 'Country']


In [84]:
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# Import the model
model = xgb.XGBRegressor()
model.load_model('attempt1_xgboost_salesmodel_online_retail.json')

# Make predictions
# Only using features in the training set
X = df[features]
predictions = model.predict(X)

In [85]:
# Adding column with predictes sales to the original dataframe
df['Predicted_Sales1'] = predictions

print(df.head())

   TransactionID InvoiceID  CustomerID InvoiceDate  Country  StockCode  \
0              1    536365        4048  2010-12-01       36       3536   
1              2    536365        4048  2010-12-01       36       2794   
2              3    536365        4048  2010-12-01       36       3044   
3              4    536365        4048  2010-12-01       36       2985   
4              5    536365        4048  2010-12-01       36       2984   

                           Description  Quantity  UnitPrice  Sales  Year  \
0   WHITE HANGING HEART T-LIGHT HOLDER         6       2.55  15.30  2010   
1                  WHITE METAL LANTERN         6       3.39  20.34  2010   
2       CREAM CUPID HEARTS COAT HANGER         8       2.75  22.00  2010   
3  KNITTED UNION FLAG HOT WATER BOTTLE         6       3.39  20.34  2010   
4       RED WOOLLY HOTTIE WHITE HEART.         6       3.39  20.34  2010   

   Month  Day  DayOfWeek  Predicted_Sales1  
0     12    1          2         14.261763  
1     12

In [87]:
#  Converting Predicted_Sales1 to float and rounding to two decimal places
df['Predicted_Sales1'] = df['Predicted_Sales1'].astype(float)
df['Predicted_Sales1'] = df['Predicted_Sales1'].round(2)

print(df.head())

   TransactionID InvoiceID  CustomerID InvoiceDate         Country  StockCode  \
0              1    536365        4048  2010-12-01  United Kingdom       3536   
1              2    536365        4048  2010-12-01  United Kingdom       2794   
2              3    536365        4048  2010-12-01  United Kingdom       3044   
3              4    536365        4048  2010-12-01  United Kingdom       2985   
4              5    536365        4048  2010-12-01  United Kingdom       2984   

                           Description  Quantity  UnitPrice  Sales  Year  \
0   WHITE HANGING HEART T-LIGHT HOLDER         6       2.55  15.30  2010   
1                  WHITE METAL LANTERN         6       3.39  20.34  2010   
2       CREAM CUPID HEARTS COAT HANGER         8       2.75  22.00  2010   
3  KNITTED UNION FLAG HOT WATER BOTTLE         6       3.39  20.34  2010   
4       RED WOOLLY HOTTIE WHITE HEART.         6       3.39  20.34  2010   

   Month  Day  DayOfWeek  Predicted_Sales1  
0     12   

In [92]:
# Filtering df so it has only InvoiceID and Predicted_Sales1 columns
df_to_insert = df[['TransactionID', 'Predicted_Sales1']]

print(df_to_insert.head())

   TransactionID  Predicted_Sales1
0              1             14.26
1              2             19.14
2              3             15.94
3              4             19.14
4              5             19.14


In [93]:
# Adding new predicted sales column to ONR_FactTransactions table in mysql

mydb, cursor = get_db_connection()

# SQL to delete the column if it exists
delete_column_sql = """
ALTER TABLE ONR_FactTransactions 
DROP COLUMN Predicted_Sales1;
"""

# SQL to add the column if it doesn't exist
add_column_sql = """
ALTER TABLE ONR_FactTransactions 
ADD COLUMN Predicted_Sales1 FLOAT;
"""

# SQL to update the comment for the new column
update_comment_sql = """
ALTER TABLE ONR_FactTransactions 
MODIFY COLUMN Predicted_Sales1 FLOAT COMMENT 'Predicted sales (UnitPrice * Quantity) with first machine learning model using XGBoost and RandomizedSearchCV.';
"""

# Execute SQL commands
try:
    cursor.execute(delete_column_sql)
    cursor.execute(add_column_sql)
    cursor.execute(update_comment_sql)
    mydb.commit()  # Commit the transaction for schema changes

    # Function to update data in MySQL
    def update_mysql(row):
        # Check if the InvoiceID exists, if yes update, else consider your strategy
        check_exists_sql = "SELECT 1 FROM ONR_FactTransactions WHERE TransactionID = %s"
        cursor.execute(check_exists_sql, (row['TransactionID'],))
        if cursor.fetchone():
            # Update existing row
            update_sql = """
            UPDATE ONR_FactTransactions 
            SET Predicted_Sales1 = %s
            WHERE InvoiceID = %s
            """
            cursor.execute(update_sql, (row['Predicted_Sales1'], row['TransactionID']))
        else:
            # Here you might want to log or handle the case where InvoiceID does not exist
            print(f"Warning: TransactionID {row['TransactionID']} does not exist in the database.")
    
    # Apply the function to each row in the dataframe
    df_to_insert.apply(update_mysql, axis=1)

    mydb.commit()  # Commit all updates
    print("Database update complete.")

except mysql.connector.Error as error:
    print(f"Failed to update database: {error}")
    mydb.rollback()  # Rollback in case of error

Failed to update database: 1060 (42S21): Duplicate column name 'Predicted_Sales1'


In [94]:
    # Close the connection
    cursor.close()
    mydb.close()